In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd

device = torch.device("cpu")

# ==============================
# 2. LOAD SVHN DATASET
# ==============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_dataset = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# ==============================
# 3. DEFINE MLP MODEL
# ==============================
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)

# ==============================
# 4. TRAIN FUNCTION
# ==============================
def train_model(optimizer_name):
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()

    if optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=0.01)
    elif optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=0.001)
    elif optimizer_name == "RMSprop":
        optimizer = optim.RMSprop(model.parameters(), lr=0.001)

    train_losses = []
    val_losses = []

    for epoch in range(5):  # keep small for Kaggle
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels % 10  # fix SVHN label issue

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_losses.append(running_loss / len(train_loader))

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels % 10

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_losses.append(val_loss / len(test_loader))
        acc = 100 * correct / total

        print(f"{optimizer_name} | Epoch {epoch+1} | Val Acc: {acc:.2f}%")

    return train_losses, val_losses, acc

# ==============================
# 5. RUN EXPERIMENTS
# ==============================
optimizers = ["SGD", "Adam", "RMSprop"]
results = {}

for opt in optimizers:
    print(f"\nRunning {opt}...\n")
    train_loss, val_loss, acc = train_model(opt)

    results[opt] = {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc
    }

# ==============================
# 6. SAVE RESULTS
# ==============================
df = pd.DataFrame({
    opt: results[opt]["val_loss"] for opt in optimizers
})

df.to_csv("optimizer_comparison.csv", index=False)

print("\nExperiment Completed. Results saved.")

100%|██████████| 182M/182M [00:21<00:00, 8.42MB/s]
100%|██████████| 64.3M/64.3M [00:14<00:00, 4.29MB/s]



Running SGD...

SGD | Epoch 1 | Val Acc: 19.59%
SGD | Epoch 2 | Val Acc: 22.22%
SGD | Epoch 3 | Val Acc: 37.81%
SGD | Epoch 4 | Val Acc: 47.12%
SGD | Epoch 5 | Val Acc: 55.42%

Running Adam...

Adam | Epoch 1 | Val Acc: 71.14%
Adam | Epoch 2 | Val Acc: 74.80%
Adam | Epoch 3 | Val Acc: 76.86%
Adam | Epoch 4 | Val Acc: 77.87%
Adam | Epoch 5 | Val Acc: 77.12%

Running RMSprop...

RMSprop | Epoch 1 | Val Acc: 61.25%
RMSprop | Epoch 2 | Val Acc: 63.06%
RMSprop | Epoch 3 | Val Acc: 74.06%
RMSprop | Epoch 4 | Val Acc: 70.82%
RMSprop | Epoch 5 | Val Acc: 76.82%

Experiment Completed. Results saved.


**SGD vs Momentum vs Nesterov (on SVHN)**

In [3]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd

device = torch.device("cpu")  # change to cuda if T4 available

# ==============================
# 2. LOAD SVHN DATASET
# ==============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_dataset = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# ==============================
# 3. DEFINE MLP MODEL
# ==============================
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)

# ==============================
# 4. TRAIN FUNCTION
# ==============================
def train_model(mode):
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()

    if mode == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=0.01)

    elif mode == "Momentum":
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)

    elif mode == "Nesterov":
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)

    train_losses = []
    val_losses = []

    for epoch in range(5):
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels % 10

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_losses.append(running_loss / len(train_loader))

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels % 10

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_losses.append(val_loss / len(test_loader))
        acc = 100 * correct / total

        print(f"{mode} | Epoch {epoch+1} | Val Acc: {acc:.2f}%")

    return train_losses, val_losses, acc

# ==============================
# 5. RUN EXPERIMENTS
# ==============================
modes = ["SGD", "Momentum", "Nesterov"]
results = {}

for m in modes:
    print(f"\nRunning {m}...\n")
    train_loss, val_loss, acc = train_model(m)

    results[m] = {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc
    }

# ==============================
# 6. SAVE RESULTS
# ==============================
df = pd.DataFrame({
    m: results[m]["val_loss"] for m in modes
})

df.to_csv("momentum_comparison.csv", index=False)

print("\nExperiment Completed. Results saved.")


Running SGD...

SGD | Epoch 1 | Val Acc: 19.63%
SGD | Epoch 2 | Val Acc: 25.22%
SGD | Epoch 3 | Val Acc: 38.76%
SGD | Epoch 4 | Val Acc: 49.36%
SGD | Epoch 5 | Val Acc: 56.80%

Running Momentum...

Momentum | Epoch 1 | Val Acc: 64.01%
Momentum | Epoch 2 | Val Acc: 72.40%
Momentum | Epoch 3 | Val Acc: 74.76%
Momentum | Epoch 4 | Val Acc: 76.66%
Momentum | Epoch 5 | Val Acc: 79.43%

Running Nesterov...

Nesterov | Epoch 1 | Val Acc: 64.14%
Nesterov | Epoch 2 | Val Acc: 73.87%
Nesterov | Epoch 3 | Val Acc: 77.08%
Nesterov | Epoch 4 | Val Acc: 76.42%
Nesterov | Epoch 5 | Val Acc: 77.45%

Experiment Completed. Results saved.


**Adagrad vs Adadelta vs Adam (SVHN)**

In [4]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd

device = torch.device("cpu")  # switch to cuda if T4 available

# ==============================
# 2. LOAD SVHN DATASET
# ==============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_dataset = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# ==============================
# 3. DEFINE MLP MODEL
# ==============================
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)

# ==============================
# 4. TRAIN FUNCTION
# ==============================
def train_model(mode):
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()

    if mode == "Adagrad":
        optimizer = optim.Adagrad(model.parameters(), lr=0.01)

    elif mode == "Adadelta":
        optimizer = optim.Adadelta(model.parameters(), lr=1.0)

    elif mode == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    val_losses = []

    for epoch in range(5):
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels % 10

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_losses.append(running_loss / len(train_loader))

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels % 10

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_losses.append(val_loss / len(test_loader))
        acc = 100 * correct / total

        print(f"{mode} | Epoch {epoch+1} | Val Acc: {acc:.2f}%")

    return train_losses, val_losses, acc

# ==============================
# 5. RUN EXPERIMENTS
# ==============================
modes = ["Adagrad", "Adadelta", "Adam"]
results = {}

for m in modes:
    print(f"\nRunning {m}...\n")
    train_loss, val_loss, acc = train_model(m)

    results[m] = {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "accuracy": acc
    }

# ==============================
# 6. SAVE RESULTS
# ==============================
df = pd.DataFrame({
    m: results[m]["val_loss"] for m in modes
})

df.to_csv("adaptive_optimizer_comparison.csv", index=False)

print("\nExperiment Completed. Results saved.")


Running Adagrad...

Adagrad | Epoch 1 | Val Acc: 64.50%
Adagrad | Epoch 2 | Val Acc: 73.21%
Adagrad | Epoch 3 | Val Acc: 71.10%
Adagrad | Epoch 4 | Val Acc: 69.35%
Adagrad | Epoch 5 | Val Acc: 78.81%

Running Adadelta...

Adadelta | Epoch 1 | Val Acc: 64.46%
Adadelta | Epoch 2 | Val Acc: 70.75%
Adadelta | Epoch 3 | Val Acc: 72.93%
Adadelta | Epoch 4 | Val Acc: 66.22%
Adadelta | Epoch 5 | Val Acc: 68.15%

Running Adam...

Adam | Epoch 1 | Val Acc: 70.44%
Adam | Epoch 2 | Val Acc: 73.53%
Adam | Epoch 3 | Val Acc: 76.48%
Adam | Epoch 4 | Val Acc: 77.17%
Adam | Epoch 5 | Val Acc: 78.93%

Experiment Completed. Results saved.


**Week-5 Dropout Experiment**

In [5]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd

device = torch.device("cpu")  # use cuda if T4 available

# ==============================
# 2. LOAD SVHN DATASET
# ==============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_dataset = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# ==============================
# 3. MODELS
# ==============================

# Without Dropout
class MLP_NoDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)


# With Dropout
class MLP_Dropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)

# ==============================
# 4. TRAIN FUNCTION
# ==============================
def train_model(model_class, name):
    model = model_class().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    train_losses = []
    val_losses = []

    for epoch in range(5):
        model.train()
        running_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels % 10

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        train_losses.append(running_loss / len(train_loader))

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels % 10

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_losses.append(val_loss / len(test_loader))
        acc = 100 * correct / total

        print(f"{name} | Epoch {epoch+1} | Val Acc: {acc:.2f}%")

    return train_losses, val_losses, acc

# ==============================
# 5. RUN EXPERIMENT
# ==============================
results = {}

print("\nRunning WITHOUT Dropout...\n")
train_loss, val_loss, acc = train_model(MLP_NoDropout, "NoDropout")
results["NoDropout"] = acc

print("\nRunning WITH Dropout...\n")
train_loss, val_loss, acc = train_model(MLP_Dropout, "Dropout")
results["Dropout"] = acc

# ==============================
# 6. SAVE RESULTS
# ==============================
df = pd.DataFrame(list(results.items()), columns=["Model", "Accuracy"])
df.to_csv("dropout_comparison.csv", index=False)

print("\nExperiment Completed. Results saved.")


Running WITHOUT Dropout...

NoDropout | Epoch 1 | Val Acc: 69.89%
NoDropout | Epoch 2 | Val Acc: 74.65%
NoDropout | Epoch 3 | Val Acc: 74.20%
NoDropout | Epoch 4 | Val Acc: 78.12%
NoDropout | Epoch 5 | Val Acc: 78.57%

Running WITH Dropout...

Dropout | Epoch 1 | Val Acc: 61.68%
Dropout | Epoch 2 | Val Acc: 66.77%
Dropout | Epoch 3 | Val Acc: 69.63%
Dropout | Epoch 4 | Val Acc: 70.28%
Dropout | Epoch 5 | Val Acc: 71.64%

Experiment Completed. Results saved.


In [6]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd

device = torch.device("cpu")  # use cuda if T4 available

# ==============================
# 2. LOAD SVHN DATASET
# ==============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_dataset = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# ==============================
# 3. MODEL (No Dropout)
# ==============================
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32*32*3, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        return self.model(x)

# ==============================
# 4. TRAIN WITH EARLY STOPPING
# ==============================
def train_early_stopping(patience=2, max_epochs=20):
    model = MLP().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    best_val_loss = float('inf')
    patience_counter = 0

    history = []

    for epoch in range(max_epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels % 10

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels % 10

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(test_loader)
        acc = 100 * correct / total

        history.append((epoch+1, train_loss, val_loss, acc))

        print(f"Epoch {epoch+1} | Val Loss: {val_loss:.4f} | Val Acc: {acc:.2f}%")

        # Early Stopping Logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_epoch = epoch + 1
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

    return history, best_epoch

# ==============================
# 5. RUN EXPERIMENT
# ==============================
history, best_epoch = train_early_stopping()

# ==============================
# 6. SAVE RESULTS
# ==============================
df = pd.DataFrame(history, columns=["Epoch", "Train Loss", "Val Loss", "Accuracy"])
df.to_csv("early_stopping_results.csv", index=False)

print(f"\nBest Epoch: {best_epoch}")
print("Experiment Completed. Results saved.")

Epoch 1 | Val Loss: 1.0132 | Val Acc: 70.14%
Epoch 2 | Val Loss: 0.8491 | Val Acc: 75.56%
Epoch 3 | Val Loss: 0.8073 | Val Acc: 77.16%
Epoch 4 | Val Loss: 0.7765 | Val Acc: 78.38%
Epoch 5 | Val Loss: 0.7765 | Val Acc: 79.01%
Epoch 6 | Val Loss: 0.8189 | Val Acc: 77.54%
Epoch 7 | Val Loss: 0.7963 | Val Acc: 78.66%

Early stopping triggered at epoch 7

Best Epoch: 5
Experiment Completed. Results saved.


In [7]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================
# 2. LOAD SVHN DATASET
# ==============================
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.SVHN(root='./data', split='train', download=True, transform=transform)
test_dataset = datasets.SVHN(root='./data', split='test', download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# ==============================
# 3. CNN MODEL
# ==============================
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 32x32
            nn.ReLU(),
            nn.MaxPool2d(2),  # 16x16

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 8x8
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8, 256),
            nn.ReLU(),
            nn.Linear(256, 10)
        )

    def forward(self, x):
        x = self.conv(x)
        x = self.fc(x)
        return x

# ==============================
# 4. TRAIN FUNCTION
# ==============================
def train_model(epochs=5):
    model = CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    history = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels % 10

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                labels = labels % 10

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_loss /= len(test_loader)
        acc = 100 * correct / total

        history.append((epoch+1, train_loss, val_loss, acc))

        print(f"Epoch {epoch+1} | Val Acc: {acc:.2f}%")

    return history

# ==============================
# 5. RUN EXPERIMENT
# ==============================
history = train_model()

# ==============================
# 6. SAVE RESULTS
# ==============================
df = pd.DataFrame(history, columns=["Epoch", "Train Loss", "Val Loss", "Accuracy"])
df.to_csv("cnn_svhn_results.csv", index=False)

print("\nExperiment Completed. Results saved.")

Epoch 1 | Val Acc: 83.54%
Epoch 2 | Val Acc: 86.33%
Epoch 3 | Val Acc: 87.59%
Epoch 4 | Val Acc: 88.18%
Epoch 5 | Val Acc: 88.77%

Experiment Completed. Results saved.
